<a href="https://colab.research.google.com/github/syedasakinashah/Sign-Bridge---Hackathon-Project/blob/main/Copy_of_hackathon_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
import os
import pandas as pd
from PIL import Image

Set up for image processing and augmentation


In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomCrop(224, padding=10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
import os

file_path = '/content/competition_data.csv.zip'

print("File size:", os.path.getsize(file_path))

In [ ]:
import pandas as pd

df = pd.read_csv('/content/competition_data.csv.zip')

print(df.head())

In [ ]:
import zipfile

zip_path = '/content/competition_data.csv.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    print(zip_ref.namelist()[:20])

Load training Data


In [ ]:
# Change this path to wherever your train folder is
full_dataset = datasets.ImageFolder('/content/competition_data.csv.zip', transform=train_transforms)

# 80/20 split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(full_dataset, [train_size, val_size])
val_data.dataset.transform = val_transforms

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

# SAVE THIS — you need it for predictions later
class_to_idx = full_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
print(idx_to_class)  # Check this looks right!

The previous cell failed because `datasets.ImageFolder` expects a directory, but you provided a `.zip` file. I will unzip the file first and then use the unzipped directory as the path for `ImageFolder`.

In [ ]:
# Unzip the dataset
import zipfile

zip_file_path = '/content/competition_data.csv.zip'
unzip_dir = '/content/competition_data'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(unzip_dir)

print(f"Dataset unzipped to: {unzip_dir}")

Load EfficientNet-B0 (pretrained, transfer learning)

In [ ]:
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# Replace the final layer for 29 classes
num_classes = 29
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

Train the model


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {running_loss/len(train_loader):.3f} | Val Accuracy: {100*correct/total:.2f}%")

Generate submission CSV


In [ ]:
# upload data file
test_folder = '/content/test'

model.eval()
results = []

for img_name in sorted(os.listdir(test_folder)):
    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    img_path = os.path.join(test_folder, img_name)
    img = Image.open(img_path).convert('RGB')
    img_tensor = val_transforms(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_tensor)
        _, predicted = torch.max(output, 1)
        label = idx_to_class[predicted.item()]

    results.append({'image_id': img_name, 'label': label})

submission = pd.DataFrame(results)
submission.to_csv('submission.csv', index=False)
print("Done! First few rows:")
print(submission.head())